# AV1 - Apoio à Decisão para Campanhas Educativas no Trânsito
## Árvore de Decisão aplicada a Sinistros de Trânsito PRF 2025

**Curso:** Sistemas de Informação  
**Disciplina:** Sistemas de Apoio à Decisão  

---

### Questão Gerencial
> *Quais características dos sinistros estão mais associadas à ocorrência de vítimas e devem ser priorizadas como foco de campanhas educativas e ações preventivas?*

### Modelo Utilizado
**Árvore de Decisão** com critério **Entropia / Ganho de Informação** (equivalente ao algoritmo ID3/C4.5).

O ganho de informação mede a redução de entropia ao particionar os dados por um atributo:

$$IG(S, A) = Entropia(S) - \sum_{x \in P(A)} \frac{|S_x|}{|S|} \cdot Entropia(S_x)$$

O atributo com **maior ganho de informação** é escolhido para dividir o nó.

---

## 0. Importações e Configurações

In [1]:
import os
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree, export_text
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, confusion_matrix,
    classification_report, ConfusionMatrixDisplay
)

warnings.filterwarnings('ignore')

# Estilo dos gráficos
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10})
sns.set_theme(style='whitegrid', palette='muted')

# Configurações do modelo
RANDOM_STATE = 42
MAX_DEPTH    = 7
MIN_SAMPLES_LEAF = 50
TEST_SIZE    = 0.20

FEATURES = [
    'dia_semana', 'fase_dia', 'condicao_metereologica',
    'tipo_pista', 'tracado_via', 'causa_acidente',
    'tipo_acidente', 'uso_solo', 'sentido_via', 'uf'
]

COLUNA_ALVO = 'classificacao_acidente'

print('Bibliotecas importadas com sucesso.')

Error: No connection selected.

---
## 1. Carregamento dos Dados

**Fonte:** Portal de Dados Abertos do Governo Federal - PRF  
**Arquivo:** `datatran2025.csv` (sinistros agrupados por ocorrência, 2025)  
**Encoding:** `latin-1` (padrão de CSVs do governo brasileiro)  
**Separador:** `;`

In [ ]:
# ── Detecta ambiente ──────────────────────────────────────────────────
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Opção A: Upload direto (abre janela para selecionar o arquivo)
    from google.colab import files
    print("Selecione o arquivo datatran2025.csv para fazer upload...")
    uploaded = files.upload()          # abre o seletor de arquivo
    CSV_PATH = list(uploaded.keys())[0]

    # Opção B (alternativa): montar o Google Drive
    # from google.colab import drive
    # drive.mount('/content/drive')
    # CSV_PATH = '/content/drive/MyDrive/datatran2025.csv'  # ajuste o caminho
else:
    # Ambiente local: busca na raiz ou em data/
    for caminho in ['datatran2025.csv', 'data/datatran2025.csv', 'data/sinistros_2025.csv']:
        if os.path.exists(caminho):
            CSV_PATH = caminho
            break
    else:
        raise FileNotFoundError("Arquivo datatran2025.csv não encontrado.")

# ── Cria pasta output se não existir ─────────────────────────────────
os.makedirs('output', exist_ok=True)

df = pd.read_csv(CSV_PATH, sep=';', encoding='latin-1', low_memory=False)

print(f'Registros : {len(df):,}')
print(f'Colunas   : {df.shape[1]}')
print(f'\nColunas disponíveis:')
print(df.columns.tolist())

In [ ]:
# Primeiras linhas do dataset
df[['dia_semana', 'fase_dia', 'causa_acidente', 'tipo_acidente',
    'condicao_metereologica', 'tipo_pista', 'classificacao_acidente']].head(8)

---
## 2. Exploração dos Dados (EDA)

### 2.1 Variável Alvo: `classificacao_acidente`

In [ ]:
# Distribuição da variável alvo
dist_alvo = df[COLUNA_ALVO].value_counts(dropna=False)
print('Distribuição da classificação dos sinistros:')
print(dist_alvo.to_string())
print(f'\nTotal com classificação válida: {dist_alvo.dropna().sum():,}')
print(f'Registros sem classificação (NaN): {df[COLUNA_ALVO].isna().sum()}')

In [ ]:
# Gráfico de distribuição da variável alvo
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Pizza
contagem = df[COLUNA_ALVO].value_counts(dropna=True)
cores = ['#e74c3c', '#e67e22', '#27ae60']
axes[0].pie(
    contagem.values, labels=contagem.index,
    autopct='%1.1f%%', colors=cores,
    startangle=140,
    wedgeprops={'edgecolor': 'white', 'linewidth': 1.5}
)
axes[0].set_title('Distribuição por Classificação', fontweight='bold')

# Barras
axes[1].barh(contagem.index, contagem.values, color=cores)
for i, v in enumerate(contagem.values):
    axes[1].text(v + 200, i, f'{v:,}', va='center', fontsize=10)
axes[1].set_xlabel('Número de Sinistros')
axes[1].set_title('Quantidade por Classificação', fontweight='bold')
axes[1].invert_yaxis()

plt.suptitle('Sinistros de Trânsito PRF 2025', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('output/distribuicao_alvo.png', dpi=150, bbox_inches='tight')
plt.show()

### 2.2 Análise das Features

In [ ]:
# Top causas de acidente
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Causa do acidente
top_causas = df['causa_acidente'].value_counts().head(10)
axes[0, 0].barh(top_causas.index[::-1], top_causas.values[::-1], color='#3498db')
axes[0, 0].set_title('Top 10 Causas de Acidente', fontweight='bold')
axes[0, 0].set_xlabel('Quantidade')

# Tipo de acidente
top_tipos = df['tipo_acidente'].value_counts().head(10)
axes[0, 1].barh(top_tipos.index[::-1], top_tipos.values[::-1], color='#e74c3c')
axes[0, 1].set_title('Top 10 Tipos de Acidente', fontweight='bold')
axes[0, 1].set_xlabel('Quantidade')

# Fase do dia
fase = df['fase_dia'].value_counts()
axes[1, 0].bar(fase.index, fase.values, color='#f39c12')
axes[1, 0].set_title('Sinistros por Fase do Dia', fontweight='bold')
axes[1, 0].set_ylabel('Quantidade')
axes[1, 0].tick_params(axis='x', rotation=15)

# Condição meteorológica
meteor = df['condicao_metereologica'].value_counts().head(7)
axes[1, 1].bar(meteor.index, meteor.values, color='#27ae60')
axes[1, 1].set_title('Sinistros por Condição Meteorológica', fontweight='bold')
axes[1, 1].set_ylabel('Quantidade')
axes[1, 1].tick_params(axis='x', rotation=20)

plt.suptitle('Análise Exploratória - Sinistros PRF 2025', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('output/analise_exploratoria.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 3. Pré-processamento

### Estratégia:
1. **Remover** registros sem classificação (evitar ruído no treinamento)
2. **Binarizar** o alvo: `1 = Com Vítimas` / `0 = Sem Vítimas`
3. **Selecionar** apenas features sem data leakage (não usar mortos, feridos, etc)
4. **Codificar** categorias com `LabelEncoder`
5. **Dividir** em treino (80%) e teste (20%) com estratificação

In [ ]:
# --- Remover registros sem classificação ---
df_clean = df.dropna(subset=[COLUNA_ALVO]).copy()
print(f'Registros originais  : {len(df):,}')
print(f'Registros utilizados : {len(df_clean):,}')
print(f'Removidos (sem alvo) : {len(df) - len(df_clean)}')

# --- Target binário ---
df_clean['com_vitimas'] = df_clean[COLUNA_ALVO].apply(
    lambda x: 1 if x in ('Com Vítimas Feridas', 'Com Vítimas Fatais') else 0
)

print(f'\nDistribuição do target binário:')
print(df_clean['com_vitimas']
      .value_counts()
      .rename({1: 'Com Vítimas (1)', 0: 'Sem Vítimas (0)'})
      .to_string())
print(f'\nDesbalanceamento: {df_clean["com_vitimas"].mean()*100:.1f}% com vítimas')

In [ ]:
# --- Codificação com LabelEncoder ---
X_raw = df_clean[FEATURES].fillna('Ignorado')
y = df_clean['com_vitimas']

encoders = {}
X_enc = X_raw.copy()
for col in FEATURES:
    le = LabelEncoder()
    X_enc[col] = le.fit_transform(X_raw[col].astype(str))
    encoders[col] = le
    print(f'{col:<32}: {le.classes_.tolist()[:4]}...' if len(le.classes_) > 4
          else f'{col:<32}: {le.classes_.tolist()}')

In [ ]:
# --- Split treino / teste estratificado ---
X_train, X_test, y_train, y_test = train_test_split(
    X_enc, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f'Treino : {len(X_train):,} registros ({int((1-TEST_SIZE)*100)}%)')
print(f'Teste  : {len(X_test):,} registros ({int(TEST_SIZE*100)}%)')
print(f'\nProporção de "Com Vítimas" no treino: {y_train.mean()*100:.1f}%')
print(f'Proporção de "Com Vítimas" no teste:  {y_test.mean()*100:.1f}%')

---
## 4. Treinamento da Árvore de Decisão

### Fundamento Teórico

O algoritmo utiliza **Entropia** como medida de impureza:

$$Entropia(S) = -\sum_{i=1}^{c} p_i \log_2 p_i$$

E seleciona a cada nó o atributo com maior **Ganho de Informação**:

$$IG(S, A) = Entropia(S) - \sum_{x \in P(A)} \frac{|S_x|}{|S|} \cdot Entropia(S_x)$$

### Hiperparâmetros
| Parâmetro | Valor | Justificativa |
|---|---|---|
| `criterion` | `'entropy'` | Usa Ganho de Informação (IG), conforme exigido |
| `max_depth` | `7` | Equilíbrio entre profundidade e overfitting |
| `min_samples_leaf` | `50` | Garante relevância estatística de cada folha |
| `random_state` | `42` | Reprodutibilidade |

In [ ]:
modelo = DecisionTreeClassifier(
    criterion='entropy',
    max_depth=MAX_DEPTH,
    min_samples_leaf=MIN_SAMPLES_LEAF,
    random_state=RANDOM_STATE
)

modelo.fit(X_train, y_train)

print('Modelo treinado com sucesso!')
print(f'  Nós na árvore     : {modelo.tree_.node_count}')
print(f'  Profundidade real : {modelo.get_depth()}')
print(f'  Folhas            : {modelo.get_n_leaves()}')

---
## 5. Avaliação do Modelo

In [ ]:
# Acurácia no treino e no teste
acc_treino = accuracy_score(y_train, modelo.predict(X_train))
acc_teste  = accuracy_score(y_test,  modelo.predict(X_test))

print(f'Acurácia no Treino : {acc_treino:.4f} ({acc_treino*100:.2f}%)')
print(f'Acurácia no Teste  : {acc_teste:.4f} ({acc_teste*100:.2f}%)')

gap = (acc_treino - acc_teste) * 100
print(f'Diferença          : {gap:.2f}%')
print(f'\nAvaliação de overfitting: {"OK - sem overfitting significativo" if gap < 3 else f"Atenção: diferença de {gap:.1f}%"}')

In [ ]:
# Relatório de classificação completo
y_pred = modelo.predict(X_test)
print('Relatório de Classificação (conjunto de teste):')
print(classification_report(
    y_test, y_pred,
    target_names=['Sem Vítimas', 'Com Vítimas']
))

In [ ]:
# Matriz de Confusão
cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

fig, ax = plt.subplots(figsize=(7, 5))
labels = ['Sem Vítimas', 'Com Vítimas']
sns.heatmap(
    cm, annot=True, fmt='d',
    xticklabels=labels, yticklabels=labels,
    cmap='Blues', ax=ax,
    linewidths=0.5, linecolor='gray',
    annot_kws={'size': 14}
)
ax.set_xlabel('Previsto pelo Modelo', fontsize=12)
ax.set_ylabel('Valor Real', fontsize=12)
ax.set_title('Matriz de Confusão - Conjunto de Teste', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.savefig('output/matriz_confusao.png', dpi=150, bbox_inches='tight')
plt.show()

print(f'\nVerdadeiros Negativos (Sem Vítimas correto) : {tn:,}')
print(f'Falsos Positivos (Sem → previsto Com)       : {fp:,}')
print(f'Falsos Negativos (Com → previsto Sem)       : {fn:,}')
print(f'Verdadeiros Positivos (Com Vítimas correto) : {tp:,}')

---
## 6. Importância das Features

A importância de cada feature é calculada como a **redução total de entropia** (impureza) ponderada pelo número de amostras que passam por cada nó onde aquele atributo foi usado para a divisão.

In [ ]:
importancias = pd.DataFrame({
    'feature': FEATURES,
    'importancia': modelo.feature_importances_
}).sort_values('importancia', ascending=False).reset_index(drop=True)

importancias['rank'] = importancias.index + 1
importancias['importancia_pct'] = importancias['importancia'] * 100

print('Ranking de Importância das Features:')
print(f'{"Rank":<5} {"Feature":<32} {"Importância":>12} {"Acumulado":>10}')
print('-' * 65)
acumulado = 0
for _, row in importancias.iterrows():
    acumulado += row['importancia_pct']
    print(f"  {int(row['rank']):<4} {row['feature']:<32} "
          f"{row['importancia_pct']:>10.2f}%  {acumulado:>8.1f}%")

In [ ]:
# Gráfico de importância
fig, ax = plt.subplots(figsize=(11, 6))
cores = ['#c0392b' if i < 3 else '#2980b9' for i in range(len(importancias))]

bars = ax.barh(
    importancias['feature'][::-1],
    importancias['importancia_pct'][::-1],
    color=cores[::-1], edgecolor='white', linewidth=0.8
)
for bar, val in zip(bars, importancias['importancia_pct'][::-1]):
    ax.text(
        bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
        f'{val:.2f}%', va='center', fontsize=10, fontweight='bold'
    )

ax.set_xlabel('Importância (% de redução de entropia)', fontsize=11)
ax.set_title(
    'Importância das Features - Árvore de Decisão\n'
    '(Redução de Entropia Ponderada por Amostras)',
    fontsize=12, fontweight='bold'
)
ax.legend(
    handles=[
        plt.Rectangle((0, 0), 1, 1, color='#c0392b', label='Top 3 - maior relevância'),
        plt.Rectangle((0, 0), 1, 1, color='#2980b9', label='Demais features'),
    ],
    loc='lower right', fontsize=9
)
plt.tight_layout()
plt.savefig('output/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 7. Visualização da Árvore de Decisão

A árvore completa tem profundidade 7. Abaixo visualizamos os primeiros **3 níveis** para legibilidade.

In [ ]:
fig, ax = plt.subplots(figsize=(30, 13))
plot_tree(
    modelo,
    feature_names=FEATURES,
    class_names=['Sem Vítimas', 'Com Vítimas'],
    filled=True,
    rounded=True,
    max_depth=3,        # exibe 3 níveis para visualização limpa
    fontsize=9,
    ax=ax,
    impurity=True       # exibe entropia de cada nó
)
ax.set_title(
    'Árvore de Decisão - Sinistros de Trânsito PRF 2025\n'
    '(Critério: Entropia / Ganho de Informação | Primeiros 3 níveis)',
    fontsize=13, fontweight='bold', pad=20
)
plt.tight_layout()
plt.savefig('output/arvore_decisao.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Regras da árvore em formato texto (primeiros 2 níveis)
print('REGRAS DA ÁRVORE (primeiros 2 níveis):')
print('='*60)
print(export_text(modelo, feature_names=FEATURES, max_depth=2))

---
## 8. Análise dos Resultados e Proposta de Campanhas Educativas

Com base nos atributos identificados pelo modelo como mais relevantes para explicar a ocorrência de vítimas nos sinistros, propõem-se as seguintes campanhas educativas:

In [ ]:
top5 = importancias.head(5)

mapa_campanhas = {
    'causa_acidente': {
        'nome': "DIRIJA COM ATENÇÃO",
        'foco': "Comportamento do condutor (distração, reação tardia, álcool, velocidade)",
        'acoes': [
            "Blitze educativas nas rodovias com maior incidência",
            "Parceria com autoescolas e aplicativos de navegação (alertas contextuais)",
            "Campanha nas redes sociais com dados reais de sinistros",
            "Educação em escolas e universidades sobre direção defensiva"
        ],
        'alvo': "Condutores em geral, especialmente jovens de 18 a 30 anos"
    },
    'tipo_acidente': {
        'nome': "DISTÂNCIA SEGURA SALVA VIDAS",
        'foco': "Prevenção de colisões traseiras, frontais e saídas de pista",
        'acoes': [
            "Painéis eletrônicos nas rodovias exibindo distância segura recomendada",
            "Fiscalização de ultrapassagem proibida com câmeras inteligentes",
            "Treinamento específico para motoristas de veículos pesados"
        ],
        'alvo': "Motoristas de caminhões, ônibus e veículos de carga"
    },
    'tracado_via': {
        'nome': "CURVAS PERIGOSAS - REDUZA A VELOCIDADE",
        'foco': "Trechos de curva, declive e aclive com maior incidência de sinistros",
        'acoes': [
            "Reforço de sinalização vertical e horizontal em curvas críticas",
            "Instalação de radares e redutores de velocidade nos pontos de risco",
            "Mapas de risco disponíveis em apps de navegação (Waze, Google Maps)"
        ],
        'alvo': "Todos os condutores em rodovias de relevo acentuado"
    },
    'fase_dia': {
        'nome': "NOITE EXIGE MAIS CUIDADO",
        'foco': "Plena noite e madrugada - menor visibilidade e maior fadiga",
        'acoes': [
            "Programas de descanso obrigatório em postos rodoviários homologados",
            "Alertas luminosos em regiões de alta incidência noturna",
            "Campanhas sobre uso de faróis e revisão de sistema de iluminação"
        ],
        'alvo': "Caminhoneiros, motoristas de aplicativo e viajantes noturnos"
    },
    'condicao_metereologica': {
        'nome': "CHUVA - REDUZA A VELOCIDADE",
        'foco': "Condições climáticas adversas: chuva, garoa, neblina",
        'acoes': [
            "Alertas em tempo real via apps e painéis de mensagem variável",
            "Treinamento sobre direção defensiva em piso molhado",
            "Campanha de verificação de pneus e sistema de freios"
        ],
        'alvo': "Todos os condutores, com destaque para regiões de clima úmido"
    },
    'tipo_pista': {
        'nome': "PISTAS SIMPLES - MÁXIMA ATENÇÃO",
        'foco': "Rodovias de pista simples com maior risco de colisão frontal",
        'acoes': [
            "Proibição reforçada de ultrapassagem com sinalização clara",
            "Demarcação de faixas de ultrapassagem segura",
            "Pressão política pela duplicação dos trechos mais críticos"
        ],
        'alvo': "Condutores e gestores de infraestrutura rodoviária"
    },
    'uso_solo': {
        'nome': "ZONA RURAL - ATENÇÃO REDOBRADA",
        'foco': "Áreas rurais onde sinistros tendem a ser mais graves",
        'acoes': [
            "Sinalização de travessias em acesso a propriedades rurais",
            "Redutores de velocidade em vias de acesso",
            "Educação de comunidades ribeirinhas às rodovias"
        ],
        'alvo': "Condutores e comunidades rurais lindeiras às rodovias"
    },
    'dia_semana': {
        'nome': "FIM DE SEMANA SEGURO",
        'foco': "Fins de semana com maior concentração de sinistros",
        'acoes': [
            "Intensificação de blitze nas sextas, sábados e domingos",
            "Campanhas contra álcool ao volante em saídas de festas e shows",
            "Parceria com bares e casas noturnas para táxis e apps de transporte"
        ],
        'alvo': "Jovens adultos e motoristas de lazer"
    },
    'sentido_via': {
        'nome': "SENTIDO CORRETO - VIDA GARANTIDA",
        'foco': "Contramão e ultrapassagens indevidas",
        'acoes': [
            "Fiscalização com câmeras de monitoramento inteligente",
            "Educação sobre regras de preferência e sinalização"
        ],
        'alvo': "Condutores em rodovias de pista simples"
    },
    'uf': {
        'nome': "CAMPANHAS REGIONALIZADAS POR ESTADO",
        'foco': "Estados com maior concentração de sinistros com vítimas",
        'acoes': [
            "Parcerias entre SENATRAN e órgãos estaduais de trânsito",
            "Campanhas adaptadas ao perfil regional (clima, relevo, frota)"
        ],
        'alvo': "Órgãos gestores e condutores por estado"
    },
}

print('=' * 70)
print('PROPOSTA DE CAMPANHAS EDUCATIVAS - BASEADA EM DADOS (DATA-DRIVEN)')
print('=' * 70)
print()

for i, row in top5.iterrows():
    feat = row['feature']
    pct  = row['importancia_pct']
    camp = mapa_campanhas.get(feat, {'nome': feat, 'foco': '-', 'acoes': [], 'alvo': '-'})
    
    print(f"{'─'*70}")
    print(f"  [{i+1}] FEATURE: {feat.upper()} - Importância: {pct:.2f}%")
    print(f"  CAMPANHA: '{camp['nome']}'")
    print(f"  FOCO    : {camp['foco']}")
    print(f"  AÇÕES   :")
    for acao in camp['acoes']:
        print(f"           • {acao}")
    print(f"  PÚBLICO : {camp['alvo']}")
    print()

---
## 9. Conclusão

### Resumo dos Resultados

O modelo de **Árvore de Decisão** baseado em **Entropia e Ganho de Informação** foi capaz de:

1. **Identificar os atributos mais relevantes** para explicar a gravidade dos sinistros (com ou sem vítimas)
2. **Gerar regras interpretáveis** que podem ser compreendidas por gestores sem conhecimento técnico aprofundado
3. **Orientar campanhas educativas** com base em evidências reais extraídas dos dados da PRF

### Vantagens do Modelo para Políticas Públicas

| Característica | Benefício |
|---|---|
| Interpretabilidade | Regras SE-ENTÃO facilmente comunicáveis |
| Identificação de padrões | Revela combinações de fatores críticos |
| Priorização de recursos | Foca campanhas nos atributos de maior impacto |
| Atualização contínua | Pode ser retreinado com novos dados a cada ano |

In [ ]:
print('Análise concluída!')
print(f'Imagens salvas na pasta output/')
print(f'\nArquivos gerados:')
for f in os.listdir('output'):
    print(f'  output/{f}')